# Bookstore Inventory and Analytics System

This notebook implements the project requirements using:

- Python control structures and arrays
- OOP with a `Bookstore` class
- NumPy for numerical calculations
- Pandas for loading, cleaning, grouping and analysis
- Matplotlib and Seaborn for visualization

**Files used:** `inventory.csv` and `sales.csv`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 1. Load the datasets

In [ ]:
inventory = pd.read_csv("inventory.csv")
sales = pd.read_csv("sales.csv")

sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["Unit_Price"] = pd.to_numeric(sales["Unit_Price"], errors="coerce")
sales["Revenue"] = pd.to_numeric(sales["Revenue"], errors="coerce")

print("Inventory shape:", inventory.shape)
print("Sales shape:", sales.shape)

display(inventory.head())
display(sales.head())


## 2. Data quality checks

In [ ]:
print("Missing values in inventory:")
display(inventory.isnull().sum().to_frame("Missing"))

print("\nMissing values in sales:")
display(sales.isnull().sum().to_frame("Missing"))

print("\nDuplicate rows:")
print("Inventory:", inventory.duplicated().sum())
print("Sales:", sales.duplicated().sum())

print("\nData types:")
display(sales.dtypes.to_frame("dtype"))


## 3. Data cleaning

In [ ]:
# Remove duplicate records
inventory = inventory.drop_duplicates().copy()
sales = sales.drop_duplicates().copy()

# Remove rows with essential missing values
inventory = inventory.dropna(subset=["Book_ID", "Title", "Price", "Quantity"]).copy()
sales = sales.dropna(
    subset=["Sale_ID", "Date", "Book_ID", "Title", "Quantity", "Revenue"]
).copy()

# Validation rules
inventory = inventory[inventory["Price"] > 0]
inventory = inventory[inventory["Quantity"] >= 0]
sales = sales[sales["Quantity"] > 0]
sales = sales[sales["Revenue"] >= 0]

# Recalculate revenue to protect against incorrect values
sales["Revenue"] = sales["Quantity"] * sales["Unit_Price"]

print("Cleaned inventory shape:", inventory.shape)
print("Cleaned sales shape:", sales.shape)


## 4. Control structures and arrays

In [ ]:
# Example: identify low-stock books using a loop and condition
low_stock_titles = []

for _, row in inventory.iterrows():
    if row["Quantity"] <= 5:
        low_stock_titles.append(row["Title"])

print("Low-stock books:")
for title in low_stock_titles:
    print("-", title)

# NumPy array calculations
revenue_array = sales["Revenue"].to_numpy(dtype=float)
quantity_array = sales["Quantity"].to_numpy(dtype=float)

print("\nNumPy total revenue:", np.sum(revenue_array))
print("NumPy average sale revenue:", np.mean(revenue_array))
print("NumPy total units sold:", np.sum(quantity_array))


## 5. OOP — Bookstore class

In [ ]:
class Bookstore:
    def __init__(self, inventory_df, sales_df):
        self.inventory = inventory_df.copy()
        self.sales = sales_df.copy()

    def add_book(self, title, author, genre, price, quantity):
        if price <= 0 or quantity < 0:
            raise ValueError("Price must be positive and quantity cannot be negative.")

        if title.lower() in self.inventory["Title"].str.lower().values:
            raise ValueError("Book already exists.")

        book_id = f"B{len(self.inventory) + 1:03d}"
        self.inventory.loc[len(self.inventory)] = [
            book_id, title, author, genre, price, quantity
        ]
        return book_id

    def update_inventory(self, title, quantity):
        if quantity < 0:
            raise ValueError("Quantity cannot be negative.")

        mask = self.inventory["Title"].str.lower() == title.lower()
        if not mask.any():
            raise ValueError("Book not found.")

        self.inventory.loc[mask, "Quantity"] = quantity

    def remove_book(self, title):
        mask = self.inventory["Title"].str.lower() == title.lower()
        if not mask.any():
            raise ValueError("Book not found.")

        self.inventory = self.inventory.loc[~mask].reset_index(drop=True)

    def record_sale(self, title, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be positive.")

        mask = self.inventory["Title"].str.lower() == title.lower()
        if not mask.any():
            raise ValueError("Book not found.")

        book = self.inventory.loc[mask].iloc[0]

        if book["Quantity"] < quantity:
            raise ValueError("Insufficient stock.")

        self.inventory.loc[mask, "Quantity"] -= quantity

        revenue = book["Price"] * quantity
        sale_id = f"S{len(self.sales) + 1:04d}"

        self.sales.loc[len(self.sales)] = [
            sale_id, pd.Timestamp.today(), book["Book_ID"], book["Title"],
            book["Author"], book["Genre"], book["Price"], quantity, revenue
        ]

        return revenue

    def generate_report(self):
        revenue = self.sales["Revenue"].to_numpy(dtype=float)
        quantity = self.sales["Quantity"].to_numpy(dtype=float)

        return {
            "Total Revenue": np.sum(revenue),
            "Total Units Sold": np.sum(quantity),
            "Average Book Price": np.mean(self.inventory["Price"].to_numpy()),
            "Low Stock Books": int((self.inventory["Quantity"] <= 5).sum())
        }

store = Bookstore(inventory, sales)
store.generate_report()


## 6. Sales analysis

In [ ]:
total_revenue = sales["Revenue"].sum()
total_units = sales["Quantity"].sum()
average_price = inventory["Price"].mean()

top_books = (
    sales.groupby("Title", as_index=False)["Quantity"]
    .sum()
    .sort_values("Quantity", ascending=False)
)

genre_sales = (
    sales.groupby("Genre", as_index=False)
    .agg(Total_Units=("Quantity", "sum"), Total_Revenue=("Revenue", "sum"))
    .sort_values("Total_Revenue", ascending=False)
)

author_sales = (
    sales.groupby("Author", as_index=False)
    .agg(Total_Units=("Quantity", "sum"), Total_Revenue=("Revenue", "sum"))
    .sort_values("Total_Revenue", ascending=False)
)

print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Units Sold: {int(total_units)}")
print(f"Average Book Price: ₹{average_price:,.2f}")

print("\nTop 10 books by units sold:")
display(top_books.head(10))

print("\nSales by genre:")
display(genre_sales)

print("\nSales by author:")
display(author_sales.head(10))


## 7. Monthly sales trend — Line Chart

In [ ]:
sales["Month"] = sales["Date"].dt.to_period("M").astype(str)

monthly_sales = (
    sales.groupby("Month", as_index=False)["Revenue"]
    .sum()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="Month", y="Revenue", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Total sales by genre — Bar Chart

In [ ]:
plt.figure(figsize=(10, 5))
genre_plot = genre_sales.sort_values("Total_Units", ascending=False)

sns.barplot(data=genre_plot, x="Genre", y="Total_Units")
plt.title("Total Units Sold by Genre")
plt.xlabel("Genre")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Top-selling books — Bar Chart

In [ ]:
top10 = top_books.head(10).sort_values("Quantity")

plt.figure(figsize=(10, 6))
sns.barplot(data=top10, x="Quantity", y="Title")
plt.title("Top 10 Best-Selling Books")
plt.xlabel("Units Sold")
plt.ylabel("Book Title")
plt.tight_layout()
plt.show()


## 10. Revenue share by genre — Pie Chart

In [ ]:
genre_revenue = sales.groupby("Genre")["Revenue"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 7))
plt.pie(
    genre_revenue.values,
    labels=genre_revenue.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Revenue Share by Book Genre")
plt.tight_layout()
plt.show()


## 11. Price vs sales volume — Scatter Plot

In [ ]:
book_analysis = (
    sales.groupby("Title", as_index=False)
    .agg(
        Unit_Price=("Unit_Price", "first"),
        Units_Sold=("Quantity", "sum"),
        Revenue=("Revenue", "sum")
    )
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=book_analysis,
    x="Unit_Price",
    y="Units_Sold",
    size="Revenue",
    sizes=(50, 500),
    alpha=0.7
)
plt.title("Book Price vs Sales Volume")
plt.xlabel("Book Price (₹)")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()


## 12. Correlation Heatmap

In [ ]:
corr_data = book_analysis[["Unit_Price", "Units_Sold", "Revenue"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_data, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Price, Sales Volume and Revenue")
plt.tight_layout()
plt.show()


## 13. Inventory status

In [ ]:
inventory["Stock_Status"] = np.where(
    inventory["Quantity"] <= 5,
    "Low Stock",
    np.where(inventory["Quantity"] <= 10, "Medium Stock", "Healthy Stock")
)

display(
    inventory[["Title", "Genre", "Price", "Quantity", "Stock_Status"]]
    .sort_values("Quantity")
    .head(15)
)

plt.figure(figsize=(8, 5))
sns.countplot(data=inventory, x="Stock_Status")
plt.title("Inventory Stock Status")
plt.xlabel("Stock Status")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.show()


## 14. Final project report

The project covers the required workflow:

1. Inventory management with validation
2. Control structures and arrays
3. `Bookstore` OOP class
4. NumPy numerical calculations
5. Pandas CSV loading, cleaning, grouping and aggregation
6. Sales analysis by book, genre, author and month
7. Matplotlib and Seaborn visualizations
8. Bar chart, line graph, pie chart, scatter plot and heatmap
